# Phishing Website Detection Using Machine Learning
**IBM SkillsBuild Cyber Security BootCamp 2026 | BharatCares**

**Student:** Dhruv  
**Project type:** Cybersecurity + Machine Learning  
**Dataset:** UCI Machine Learning Repository — Phishing Websites (Dataset ID: 327)

## Project overview
Phishing websites imitate trusted websites to steal credentials or other sensitive information. This notebook trains and evaluates machine-learning classifiers using the UCI Phishing Websites dataset.

**Important scope note:** The UCI dataset contains 30 pre-extracted website/URL security features, not raw URL strings. Therefore, this notebook predicts from a row of those 30 features. It does not claim to safely inspect or classify an arbitrary URL entered as plain text.

## Objectives
1. Load a publicly available phishing website dataset.
2. Inspect and prepare the feature data.
3. Train machine-learning classification models.
4. Evaluate the models using accuracy, precision, recall, F1-score and a confusion matrix.
5. Demonstrate a prediction using a held-out test record.

**Safety:** This is an educational demonstration, not a production security scanner. Never visit suspicious links just to test them.

## 1. Install and import dependencies

Install the libraries listed in `requirements.txt` before running the notebook. If you are using Google Colab, uncomment and run the installation cell below.

In [ ]:
# Uncomment this line if the required libraries are not already installed:
# %pip install -r requirements.txt

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 2. Load the dataset

The notebook retrieves dataset ID **327** from the UCI Machine Learning Repository using `ucimlrepo`. An internet connection is required for the first download.

In [ ]:
phishing_dataset = fetch_ucirepo(id=327)

X = phishing_dataset.data.features.copy()
y = phishing_dataset.data.targets.copy()

# The UCI target is normally a one-column DataFrame named 'Result'.
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\\nFirst five feature rows:")
display(X.head())
print("\\nTarget distribution:")
display(y.value_counts(dropna=False))

## 3. Understand and prepare the data

The dataset contains 11,055 records and 30 integer-valued features. The original target convention is **-1 = phishing** and **1 = legitimate**. The code below standardizes labels to readable class names while preserving the feature values.

In [ ]:
# Ensure the feature columns are numeric.
X = X.apply(pd.to_numeric, errors="coerce")

# Convert target labels into readable names.
y_numeric = pd.to_numeric(y, errors="coerce")
label_map = {-1: "Phishing", 1: "Legitimate"}
y = y_numeric.map(label_map)

# Remove rows with missing feature values or unknown target labels.
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

print("Clean feature shape:", X.shape)
print("Missing feature values:", int(X.isna().sum().sum()))
print("\\nClass counts:")
display(y.value_counts())

In [ ]:
# Dataset summary
display(X.describe().T)

# Class distribution chart
plt.figure(figsize=(7, 4))
sns.countplot(x=y, order=["Phishing", "Legitimate"])
plt.title("Distribution of Website Classes")
plt.xlabel("Website class")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 4. Split the dataset

A stratified 80:20 train-test split keeps the class proportions approximately consistent in both sets. The test data is held back for final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Number of input features:", X_train.shape[1])

## 5. Train machine-learning models

Two supervised classification algorithms are trained:
- **Decision Tree:** learns decision rules from feature values.
- **Random Forest:** combines multiple decision trees to improve robustness.

The models are trained only on the training split.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=12
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{model_name} training completed.")

## 6. Evaluate model performance

Accuracy measures overall correct predictions. Precision measures how many predicted phishing/legitimate labels are correct for each class; recall measures how many actual records of a class are found. The F1-score balances precision and recall.

For cybersecurity, **recall for the phishing class is particularly important**, because missed phishing sites can be risky.

In [ ]:
results = []

for model_name, model in models.items():
    predictions = model.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (macro)": precision_score(y_test, predictions, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, predictions, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, predictions, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))

In [ ]:
# Select the best model by macro F1-score on the held-out test set.
best_model_name = results_df["F1-score (macro)"].idxmax()
best_model = models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Selected model:", best_model_name)
print("\\nClassification report:")
print(classification_report(
    y_test,
    best_predictions,
    labels=["Phishing", "Legitimate"],
    zero_division=0
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(
    y_test,
    best_predictions,
    labels=["Phishing", "Legitimate"]
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Phishing", "Legitimate"],
    yticklabels=["Phishing", "Legitimate"]
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()

## 7. Feature importance

Random Forest provides a feature-importance score indicating how useful each feature was to the model's tree splits. These scores describe model behavior; they do not prove that a feature causes a website to be malicious.

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(
        best_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)

    display(importance.head(15).to_frame("Importance"))

    plt.figure(figsize=(9, 6))
    importance.head(15).sort_values().plot(kind="barh")
    plt.title(f"Top 15 Feature Importances — {best_model_name}")
    plt.xlabel("Importance score")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance is not available for the selected model.")

## 8. Demonstrate a prediction

The example below selects one record from the held-out test set and asks the trained model to classify its 30 pre-extracted features. This demonstrates the prediction pipeline without inventing feature values or pretending that a plain URL string is sufficient input.

In [ ]:
# Use a real held-out test record for the demonstration.
sample_features = X_test.iloc[[0]]
actual_class = y_test.iloc[0]
predicted_class = best_model.predict(sample_features)[0]

print("Actual class:", actual_class)
print("Predicted class:", predicted_class)

if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(sample_features)[0]
    probability_table = pd.DataFrame({
        "Class": best_model.classes_,
        "Estimated probability": probabilities
    }).sort_values("Estimated probability", ascending=False)
    display(probability_table)

display(sample_features)

## 9. Conclusion

This project demonstrates an end-to-end machine-learning workflow for phishing website classification using a public UCI dataset. It includes data loading, inspection, train-test splitting, model training, evaluation and a sample prediction.

### Limitations
- The dataset is a benchmark dataset and may not represent every current phishing technique.
- Input features are already extracted and encoded; the notebook does not extract them from a live website.
- Results depend on the dataset, split and model settings. Report the metrics generated by your own run; do not claim a fixed accuracy in advance.
- A model prediction is not a guarantee that a website is safe. Use trusted security tools and verify suspicious links independently.

### Dataset citation
Mohammad, R., McCluskey, L., & Thabtah, F. (2012). *Phishing Websites*. UCI Machine Learning Repository. https://doi.org/10.24432/C51W2X

Dataset page: https://archive.ics.uci.edu/dataset/327/phishing

### Final note
Run all cells from top to bottom before submission. Check that the output tables and charts are visible, and save the notebook with its executed outputs if your submission portal accepts it.